In [ ]:
from pathlib import Path
import os

repo_folder = "/home/automl/git/iot-threat-classifier"

base_folder = Path(os.path.join(repo_folder), "2025-11-17/Input_Zip_v3b")

dispatcher_filename = Path(os.path.join(base_folder, "start"))

In [ ]:
BINARIZE_FLAGS = [False]
SEEDS = [17, 23, 37, 53, 89]
SAMPLE_FRACS = [0.05, 0.10, 0.25, 0.50] # Do not put "full" here!!!
SAMPLE_FRACS_STR = [f"{int(100 * sf):02}" for sf in SAMPLE_FRACS]

In [ ]:
from datetime import datetime
from time import sleep

def now():
    now = datetime.now()
    yyyymmdd_hhmmss_part = now.strftime('%Y-%m-%d %H:%M:%S')
    ms_part = f'{int(now.microsecond / 1000):03d}'
    return f'{yyyymmdd_hhmmss_part},{ms_part}'

# while not dispatcher_filename.exists():
#     print(f'[{now()}] Dispatcher file does not exist; sleeping...')
#     sleep(300)

print(f'[{now()}] Dispatcher file EXISTS; starting...')

In [ ]:
import numpy as np
import pandas as pd

def get_dir_size(path):
    root = Path(path)
    return sum(f.stat().st_size for f in root.rglob('*') if f.is_file())

def load_results(results_filename):
    df = pd.read_excel(results_filename)
    try:
        row = df.loc[df['f1_score_abs'].idxmax(), ['Unnamed: 0', 'f1_score_abs']]
        max_config = str(row['Unnamed: 0'])
        max_f1_weighted = f"{float(row['f1_score_abs']):.6f}"
    except Exception:
        max_config = None
        max_f1_weighted = str(np.nan)
    cfg_str = f"{max_config if max_config else 'N/A':<8}"
    f1_str = f"{max_f1_weighted if max_f1_weighted not in ['nan', None] else 'N/A':<8}"
    return cfg_str, f1_str

def parse_exception(e):
    try:
        return str(e).split("\n")[-2]
    except:
        return "unknown error"

In [ ]:
import json

dataset_info = {}

# Find all instances of metadata.json recursively
found_files = base_folder.rglob('*.metadata.json')

# For testing purposes
# found_files = [x for x in found_files if 'KDD' in str(x)]

for file_path in found_files:
    with open(file_path, mode='r', encoding='utf-8') as fp:
        metadata = json.load(fp)

    dataset_name = metadata['name']
    dataset_folder_suffix = 'Binary' if metadata['binarize'] else 'Multiclass'
    dataset_folder_path = os.path.join(base_folder, metadata['name'], dataset_folder_suffix)
    dataset_folder_size = get_dir_size(dataset_folder_path)
    dataset_config = f"B:{metadata['binarize']};S:{metadata['seed']};F:{int(100 * metadata['sample_frac']):02}"

    # subset_prefix = "Binary" if metadata['binarize'] else 'Multiclass'
    subset_prefix = f"seed_{metadata['seed']}"
    subset_prefix += f"/sampled_{int(100 * metadata['sample_frac']):02}"
    subset_prefix = subset_prefix.replace("sampled_100", "full")

    subset_suffixes = ['train', 'val', 'test']
    subset_exists = {s: False for s in subset_suffixes}
    for subset_suffix in subset_suffixes:
        subset_filename = os.path.join(dataset_folder_path, subset_prefix, f"{dataset_name}_{subset_suffix}.parquet")
        subset_exists[subset_suffix] = Path(subset_filename).is_file()
    subset_exists_str = ";".join([f"{k.title()}:{str(v)}" for k, v in subset_exists.items()])

    if dataset_name not in dataset_info.keys():
        dataset_info[dataset_name] = {
            'path': dataset_folder_path,
            'size': dataset_folder_size,
            'configs': set()
        }
    dataset_info[dataset_name]['configs'].add(f"{subset_exists_str};{dataset_config}")

In [ ]:
import itertools

# Create all combinations once
all_configs = {f"Train:True;Val:True;Test:True;B:{b};S:{s};F:{f}"
               for b, s, f in itertools.product(BINARIZE_FLAGS, SEEDS, SAMPLE_FRACS_STR)}

# Sort the dataset info
sorted_dataset_info = dict(sorted(
    dataset_info.items(), 
    key=lambda item: item[1]['size']
))

# Dictionary to hold processed data (for later processing)
ready_datasets = {}

for k, v in sorted_dataset_info.items():
    input_path = v['path']
    output_path = v['path'].replace('Input', 'Output')
    size_mb = v['size'] / (1024**2)
    is_ready = v['configs'].issuperset(all_configs)
    
    if is_ready:
        status = 'ready'
        details = "----"
    else:
        status = 'missing'
        # Calculate missing configs
        missing = all_configs - set(v['configs'])
        # Convert set to string or list for display
        details = str(missing)

    # Store in dict keyed by dataset name
    ready_datasets[k] = {
        'input_path': input_path,
        'output_path': output_path,
        'size_mb': size_mb,
        'status': status,
        'details': details
    }

# Create the DataFrame from the dictionary
df = pd.DataFrame.from_dict(ready_datasets, orient='index')
df.index.name = 'name'
df = df.reset_index()
df['input_path'] = df['input_path'].apply(lambda x: x.replace(repo_folder, ''))
df['output_path'] = df['output_path'].apply(lambda x: x.replace(repo_folder, ''))
df['size_mb'] = df['size_mb'].astype(float)

# Configure Pandas to display all rows and full column content
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.2f}'.format)

styled_df = df.style.set_properties(subset=['name', 'input_path', 'output_path'], **{'text-align': 'left'}) \
                    .format({'size_mb': "{:.2f}"}) \
                    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])

display(styled_df)

In [ ]:
from tqdm.auto import tqdm
import papermill as pm

# 1. Filter out datasets with missing data
ready_datasets = {k: v for k, v in ready_datasets.copy().items() if v['status'] == 'ready'}

# 2. Master Job List
job_queue = list(itertools.product(
    ready_datasets,
    BINARIZE_FLAGS, 
    SEEDS,
    SAMPLE_FRACS
))

errors = {}

# 3. Progress Bar
pbar = tqdm(job_queue, desc='Progress')

for ds_name, binarize, seed, sample_frac in pbar:
    
    # Description to know what is running
    seed_part = f"seed_{seed}"
    frac_part = f"sampled_{int(100 * sample_frac):02}"
    frac_part_short = frac_part.replace("sampled_", "")
    pbar.set_postfix({"DS": ds_name, "B": str(binarize), "S": seed, "F": frac_part_short})

    # Path construction
    ds_input_path = ready_datasets[ds_name]['input_path']
    ds_output_path = os.path.join(ready_datasets[ds_name]['output_path'], f"{seed_part}/{frac_part}")
    os.makedirs(ds_output_path, exist_ok=True)

    input_notebook = 'evaluator_code_v2.ipynb'
    output_notebook = os.path.join(ds_output_path, 'xgb_execution.ipynb')
    results_filename = os.path.join(ds_output_path, 'xgb_summary_table.xlsx')
    params_file = output_notebook.replace('.ipynb', '_params.json')
    error_file = output_notebook.replace('.ipynb', '_errors.json')

    # Initialize parameters to None to avoid error handler from crashing
    parameters = None

    try:
        # Skip if already done
        if Path(results_filename).exists():
            continue

        parameters = dict(
            dataset_name=ds_name,
            binarize=binarize,
            random_state=seed,
            sample_frac=sample_frac,
            target_column='label',
            min_samples_per_class=1,
            feature_selection_threshold=0.95,
            sample_filtering_quantile=0.10,
            hpo_n_trials=10, 
            hpo_timeout=60, 
            num_boost_round=100,
            early_stopping_rounds=10,
            plot_param_importances=False,
            n_jobs=-1
        )

        # Save params before execution to debug input if it crashes
        with open(params_file, 'w', encoding='utf-8') as f:
            json.dump(parameters, f, indent=4)                
    
        # Execute the job
        pm.execute_notebook(
            input_notebook, 
            output_notebook, 
            parameters=parameters,
            kernel_name='python3',
            log_output=False
        )
        
        # Load results
        max_config, max_f1_weighted = load_results(results_filename)
        tqdm.write(f'[{now()}] SUCCESS | DS = {ds_name} | B = {binarize} | S = {seed} | F = {frac_part_short} | F1: {max_f1_weighted}')

    except Exception as e:

        print(e)
        
        tqdm.write(f'[{now()}] ERROR   | DS = {ds_name} | B = {binarize} | S = {seed} | F = {frac_part_short} | {parse_exception(e)}')
        
        error_json = {
            "timestamp": now(), 
            "parameters": parameters if parameters else "Not generated", 
            "error": str(e).split('\n')
        }
        
        with open(error_file, 'w', encoding='utf-8') as f:
            json.dump(error_json, f, indent=4)